# Learned Bertsekas Auction

Fully decentralised drone-to-slot assignment. No Hungarian or Sinkhorn in the inference path.

| Component | Role |
|---|---|
| `LearnedBertsekaGNN` | Encoder producing per-drone-slot values V[i,j] and per-slot ε_j |
| `bertsekas_price_auction` | Decentralised Bertsekas protocol — price memory, comm-graph propagation |
| `bertsekas_loss` | CE on V + cost-regret + ε calibration |

**Why this is better than the previous decentralised model:**  
The previous auction blocked lost slots but had no shared price memory — drone 12 didn't know slot 3 was contested until it tried and lost. Here, slot prices accumulate and propagate through the comm graph each round. Every drone self-selects away from expensive slots *before* bidding, which is exactly what makes Bertsekas converge to near-optimal.

Use the `torch_env` kernel.

In [12]:
import os, random
import numpy as np
import torch

from local_negotiator import (
    LocalNegotiatorGNN,
    evaluate_strict_decentralized,
    load_negotiator_dataset,
    prepare_dataset,
)
from bertsekas_auction_v2 import (
    LearnedBertsekaModel,
    evaluate_bertsekas,
    train_bertsekas_model,
    AUCTION_ROUNDS,
)

SEED         = 42
DATASET      = './dataset/negotiator_dataset_v1.pt'
BASE_CKPT    = 'strict_local_negotiator_best_v1.pt'
OUT          = 'bertsekas_best.pt'

EPOCHS       = 80
LR           = 3e-4
FREEZE_EPOCHS = 10
MAX_TRAIN    = 7000
MAX_VAL      = 500
MAX_TEST     = 500

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}  |  Torch: {torch.__version__}')

Device: cuda  |  Torch: 2.10.0+cu128


In [13]:
# ── Load base checkpoint ──────────────────────────────────────────────────────
base = LocalNegotiatorGNN().to(DEVICE)
if os.path.isfile(BASE_CKPT):
    ckpt = torch.load(BASE_CKPT, map_location='cpu', weights_only=False)
    base.load_state_dict(ckpt['model_state_dict'])
    base.to(DEVICE)
    print('Loaded base checkpoint:')
    print(f"  gossip bijection : {ckpt['metrics']['test']['bijection_rate']:.3f}")
    print(f"  gossip cost ratio: {ckpt['metrics']['test']['cost_ratio_vs_hungarian']:.4f}")
    print(f"  gossip match     : {ckpt['metrics']['test']['slot_match_rate']:.3f}")
else:
    print('No base checkpoint — training from scratch')

model = LearnedBertsekaModel(base).to(DEVICE)
total_p = sum(p.numel() for p in model.parameters())
new_p   = sum(p.numel() for p in [
    *model.gnn.price_proj.parameters(),
    *model.gnn.epsilon_head.parameters(),
])
print(f'\nTotal params : {total_p:,}')
print(f'New params   : {new_p:,}  (frozen base: {total_p - new_p:,})')

Loaded base checkpoint:
  gossip bijection : 0.589
  gossip cost ratio: 1.0340
  gossip match     : 0.640

Total params : 34,659
New params   : 4,033  (frozen base: 30,626)


In [14]:
# ── Dataset ───────────────────────────────────────────────────────────────────
raw = load_negotiator_dataset(DATASET)
train_data, val_data, test_data = prepare_dataset(
    raw,
    model.formation_embedding.weight.detach().cpu(),
    seed=SEED, force_gt_visibility=False,
)
train_data = train_data[:MAX_TRAIN]
val_data   = val_data[:MAX_VAL]
test_data  = test_data[:MAX_TEST]
print(f'Train={len(train_data)}  Val={len(val_data)}  Test={len(test_data)}')

Train=7000  Val=500  Test=500


In [15]:
# ── Baselines ─────────────────────────────────────────────────────────────────
print('=== Baselines (test set, before training) ===')

print('\nGossip consensus (base model):')
gossip = evaluate_strict_decentralized(base, test_data, DEVICE)
for k, v in gossip.items(): print(f'  {k}: {v:.4f}')

print('\nBertsekas auction (untrained values, learned ε):')
baseline = evaluate_bertsekas(model, test_data[:200], DEVICE)
for k, v in baseline.items(): print(f'  {k}: {v:.4f}')

=== Baselines (test set, before training) ===

Gossip consensus (base model):
  bijection_rate: 0.5400
  conflict_rate: 0.0266
  unassigned_rate: 0.0159
  cost_ratio_vs_hungarian: 1.0388
  slot_match_rate: 0.6398
  consensus_rounds: 9.4880
  converged_rate: 0.5580

Bertsekas auction (untrained values, learned ε):
  bijection_rate: 0.4300
  conflict_rate: 0.0000
  unassigned_rate: 0.0564
  cost_ratio_vs_hungarian: 1.0603
  slot_match_rate: 0.6242
  consensus_rounds: 1.1800
  converged_rate: 1.0000
  messages_sent: 103.0450


In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
print('=== Learned Bertsekas training ===')
history = train_bertsekas_model(
    model,
    train_data,
    val_data,
    DEVICE,
    epochs=EPOCHS,
    lr=LR,
    freeze_epochs=FREEZE_EPOCHS,
    force_gt_train=True,
)

=== Learned Bertsekas training ===
Epoch   1 [frozen] | train=0.7615 | val=0.8202 | bij=0.440 | conflict=0.000 | cost=1.0615 | match=0.659 | conv=1.000 | rounds=1.1


In [ ]:
# ── Final evaluation ──────────────────────────────────────────────────────────
print('=== Final evaluation (test set) ===')
final = evaluate_bertsekas(model, test_data, DEVICE)

gossip_final = evaluate_strict_decentralized(base, test_data, DEVICE)

keys = ['bijection_rate', 'cost_ratio_vs_hungarian', 'slot_match_rate',
        'conflict_rate', 'unassigned_rate', 'consensus_rounds']
print(f'{"metric":<35} {"gossip (base)":>16} {"bertsekas":>12}')
print('-' * 65)
for k in keys:
    g = gossip_final.get(k, 0.0)
    b = final.get(k, 0.0)
    better = ''
    if k in ('bijection_rate', 'slot_match_rate') and b > g: better = ' ↑'
    if k in ('cost_ratio_vs_hungarian', 'conflict_rate',
              'unassigned_rate', 'consensus_rounds') and b < g: better = ' ↓'
    print(f'{k:<35} {g:>16.4f} {b:>12.4f}{better}')

=== Final evaluation (test set) ===
metric                                 gossip (base)    bertsekas
-----------------------------------------------------------------
bijection_rate                                0.5260       0.0000
cost_ratio_vs_hungarian                       1.0464       2.3438
slot_match_rate                               0.5858       0.0817
conflict_rate                                 0.0294       0.0000 ↓
unassigned_rate                               0.0185       0.6931
consensus_rounds                              9.7920      20.0000


In [ ]:
# ── Save ──────────────────────────────────────────────────────────────────────
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'base_checkpoint':  BASE_CKPT,
        'epochs':           EPOCHS,
        'lr':               LR,
        'freeze_epochs':    FREEZE_EPOCHS,
        'auction_rounds':   AUCTION_ROUNDS,
        'inference':        'fully decentralised Bertsekas price auction; no Hungarian/Sinkhorn',
    },
    'history': history,
    'metrics': {
        'bertsekas': final,
        'gossip_baseline': gossip_final,
    },
}, OUT)
print(f'Saved → {OUT}')

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Learned Bertsekas Auction training', fontsize=13, fontweight='bold')
ep = range(1, len(history['train_loss']) + 1)

axes[0].plot(ep, history['train_loss'], label='train', color='steelblue')
axes[0].plot(ep, history['val_loss'],   label='val',   color='coral', linestyle='--')
axes[0].axvline(FREEZE_EPOCHS, color='gray', linestyle=':', alpha=0.7, label='unfreeze')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep, history['val_bijection_rate'], label='bijection', color='green')
axes[1].plot(ep, history['val_slot_match_rate'], label='match',    color='purple', linestyle='--')
axes[1].axvline(FREEZE_EPOCHS, color='gray', linestyle=':', alpha=0.7)
axes[1].set_title('Bijection & Match Rate'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(ep, history['val_cost_ratio_vs_hungarian'], label='cost ratio', color='orange')
axes[2].axvline(FREEZE_EPOCHS, color='gray', linestyle=':', alpha=0.7)
axes[2].axhline(1.0, color='gray', linestyle='-', alpha=0.3, label='optimal')
axes[2].set_title('Cost Ratio vs Hungarian'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('bertsekas_training_curves.png', dpi=120)
plt.show()
print('Saved bertsekas_training_curves.png')